<a href="https://colab.research.google.com/github/harigandan/GEN-AI-LAB-EXERSISE/blob/main/gen_ai_%26_llm_ex_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer, AutoModelForQuestionAnswering

# Ensure torch is installed and available

# ---------- Text Summarization ----------
# The pipeline function can directly handle summarization, simplifying the code.
# Fix: Explicitly load model and tokenizer for summarization, bypassing pipeline task validation.
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

article = """Generative AI refers to a class of artificial intelligence models capable of
producing new content such as text, images, audio, and video. Large Language Models (LLMs)
such as GPT and LLaMA are trained on massive text corpora and can perform a wide range of
natural language tasks including translation, summarization, and question answering. These
models are increasingly being deployed in industry applications ranging from customer support
to software development, transforming how humans interact with machines."""

# Prepare inputs for the model
inputs = tokenizer([article], max_length=1024, return_tensors="pt", truncation=True)

# Generate Summary
summary_ids = model.generate(
    inputs["input_ids"],
    num_beams=4,
    min_length=20,
    max_length=45,
    early_stopping=True
)
summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Summary:\n", summary_text)

# ---------- Question Answering ----------
# Fix: Explicitly load model and tokenizer for question-answering, bypassing pipeline task validation.
qa_model_name = "distilbert-base-cased-distilled-squad"
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_name)

context = article
question = "What are Large Language Models trained on?"

try:
    # Fix: Use direct call to tokenizer for encoding with truncation
    inputs = qa_tokenizer(question, context, return_tensors="pt", truncation=True)
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    with torch.no_grad():
        outputs = qa_model(input_ids, attention_mask=attention_mask)

    answer_start_scores = outputs.start_logits
    answer_end_scores = outputs.end_logits

    # Get the most likely beginning and end of the answer span
    answer_start = torch.argmax(answer_start_scores)
    answer_end = torch.argmax(answer_end_scores) + 1 # +1 because slicing is exclusive

    answer_text = qa_tokenizer.decode(input_ids[0][answer_start:answer_end])

    # Calculate a simplified confidence score
    start_scores_softmax = torch.softmax(answer_start_scores, dim=1)
    end_scores_softmax = torch.softmax(answer_end_scores, dim=1)
    confidence_score = start_scores_softmax[0, answer_start].item() * end_scores_softmax[0, answer_end - 1].item()

    print("\nQuestion:", question)
    print("Answer:", answer_text, "| Confidence:", round(confidence_score, 3))
except Exception as e:
    print("\nError during Question Answering:", e)
    import traceback
    traceback.print_exc()

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Summary:
 Large Language Models (LLMs) are trained on massive text corpora. They can perform a wide range of natural language tasks including translation, summarization, and question answering.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]


Question: What are Large Language Models trained on?
Answer: massive text corpora | Confidence: 0.892
